## **HRRR Data Integration**


### **Required Libraries**

In [60]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, date, timezone, timedelta
import requests
import random
from tqdm import tqdm
import re

from eccodes import (
    codes_get,
    codes_get_array,
    codes_grib_new_from_file,
    codes_release,
    )




### **Root Paths**

In [8]:

root_path = Path.cwd()
data_path = root_path / 'data'
data_path.mkdir(exist_ok=True)
print(data_path.exists())


True


### **Utils**

In [81]:


# a function to download a single file
def download_file(url, destination, chunk_size=8 * 1024 * 1024):
    """Download a file with size verification and tqdm progress."""

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = destination.with_suffix(destination.suffix + ".part")

    # what if the file already exists - lets skip it in that case
    # we will stream response to access headers without downloading the body
    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()

        # lets get expected file size from server and later match with the file if it exists
        expected_size = int(response.headers.get("Content-Length", 0))

        # now skip only if existing file size matches expected size
        if destination.exists() and expected_size > 0:
            existing_size = destination.stat().st_size

            if existing_size == expected_size:
                print(f"Already downloaded: {destination.name} "
                      f"({existing_size / 1e6:.1f} MB)")
                return destination

            print(f"Size mismatch: {destination.name} "
                  f"(local={existing_size:,}, expected={expected_size:,})")
            print("Re-downloading...")

        # lets also add download with progress bar
        with temporary_path.open("wb") as output_file:
            with tqdm(
                total=expected_size or None,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=destination.name
                ) as progress:

                for chunk in response.iter_content(chunk_size=chunk_size):
                    if chunk:
                        output_file.write(chunk)
                        progress.update(len(chunk))

    # verify downloaded size and again match with the expected size
    downloaded_size = temporary_path.stat().st_size

    if expected_size and downloaded_size != expected_size:
        raise IOError(
            f"Incomplete download for {destination.name}: "
            f"expected {expected_size:,} bytes, "
            f"received {downloaded_size:,}"
            )

    # replace final file only after successful download
    temporary_path.replace(destination)

    print(f"Downloaded: {destination.name} "
          f"({downloaded_size / 1e6:.1f} MB)")

    return destination



def inspect_downloaded_file(grib_path):
    """
    Inspect the identity and basic integrity of one downloaded HRRR file.

    """
    
    grib_path = Path(grib_path)
    
    # get the index path
    index_path = Path(str(grib_path) + ".idx")

    # lets get the pattern and match that pattern and use the filename to get the infor such as
    # cycle hour, lead hour and initialize time
    filename_pattern = re.compile(r"hrrr\.t(?P<cycle>\d{2})z\.wrfsfcf(?P<lead>\d{2})\.grib2$")
    
    filename_match = filename_pattern.match(grib_path.name)

    if filename_match is None:
        raise ValueError(f"Unexpected HRRR filename: {grib_path.name}")

    # now lets get the cycle hour, lead hour and intialization time
    cycle_hour = int(filename_match.group("cycle"))
    lead_hour = int(filename_match.group("lead"))

    # get the initialization time - it should be our download_date
    initialization_time = datetime.strptime(
                            f"{download_date}{cycle_hour:02d}",
                            "%Y%m%d%H").replace(tzinfo=timezone.utc)


    # valid time - if it is f0, then valid tiem will be simply one hour lead time
    valid_time = initialization_time + timedelta(hours=lead_hour)
    
    source_url = hrrr_url(download_date, cycle_hour, lead_hour)

    grib_exists = grib_path.exists()
    index_exists = index_path.exists()

    grib_size = grib_path.stat().st_size if grib_exists else None
    index_size = index_path.stat().st_size if index_exists else None

    starts_with_grib = False
    ends_with_7777 = False

    if grib_exists and grib_size >= 8:
        with grib_path.open("rb") as file:
            starts_with_grib = file.read(4) == b"GRIB"
            file.seek(-4, 2)
            ends_with_7777 = file.read(4) == b"7777"

    index_line_count = None

    if index_exists:
        with index_path.open("r", encoding="utf-8") as file:
            index_line_count = sum(1 for line in file if line.strip())

    return {
        "filename": grib_path.name,
        "cycle_hour_utc": cycle_hour,
        "forecast_hour": lead_hour,
        "initialization_time": initialization_time,
        "valid_time": valid_time,
        "grib_exists": grib_exists,
        "index_exists": index_exists,
        "grib_size_mb": grib_size / 1e6 if grib_size else None,
        "index_size_kb": index_size / 1e3 if index_size else None,
        "index_message_count": index_line_count,
        "starts_with_grib": starts_with_grib,
        "ends_with_7777": ends_with_7777,
        "basic_integrity_passed": (
                                grib_exists
                                and index_exists
                                and starts_with_grib
                                and ends_with_7777
                                ),
        "source_url": source_url,
        "local_path": str(grib_path.resolve()),
        }


def parse_hrrr_index(index_path, grib_path):
    """
    Convert an HRRR index file into a searchable table.

    """
    
    records = []

    with Path(index_path).open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line: continue

            parts = line.split(":")

            records.append(
                    {
                    "message": int(parts[0]),
                    "start_byte": int(parts[1]),
                    "reference_time": parts[2].removeprefix("d="),
                    "variable": parts[3],
                    "level": parts[4],
                    "time_description": parts[5],
                    "original_index_line": line,
                    }
                    )

    index_table = pd.DataFrame(records)
    index_table = index_table.sort_values("start_byte").reset_index(drop=True)

    file_size = Path(grib_path).stat().st_size

    index_table["end_byte"] = (
        index_table["start_byte"]
        .shift(-1)
        .fillna(file_size)
        .astype("int64")
        - 1
        )

    index_table["message_size_mb"] = (index_table["end_byte"] - index_table["start_byte"] + 1) / 1e6

    return index_table




def safe_grib_get(message, key):
    """
    Read a GRIB key and return None if the key is unavailable.

    """
    
    try:
        return codes_get(message, key)
    except Exception:
        return None


def read_selected_messages(grib_path, selected_records):
    """
    Read metadata and values for selected GRIB messages.

    """
    message_to_field = dict(zip(selected_records["message"], selected_records["field_name"]))

    selected_data = {}

    with Path(grib_path).open("rb") as file:
        message_number = 0

        while True:
            message = codes_grib_new_from_file(file)

            if message is None:
                break

            message_number += 1

            if message_number in message_to_field:
                field_name = message_to_field[message_number]

                
                metadata = {
                    "field_name": field_name,
                    "name": safe_grib_get(message, "name"),
                    "short_name": safe_grib_get(message, "shortName"),
                    "units": safe_grib_get(message, "units"),
                    "type_of_level": safe_grib_get(message, "typeOfLevel"),
                    "level": safe_grib_get(message, "level"),
                    "step_type": safe_grib_get(message, "stepType"),
                    "start_step": safe_grib_get(message, "startStep"),
                    "end_step": safe_grib_get(message, "endStep"),
                    "initialization_date": safe_grib_get(message, "dataDate"),
                    "initialization_time": safe_grib_get(message, "dataTime"),
                    "validity_date": safe_grib_get(message, "validityDate"),
                    "validity_time": safe_grib_get(message, "validityTime"),
                    "grid_type": safe_grib_get(message, "gridType"),
                    "nx": safe_grib_get(message, "Nx"),
                    "ny": safe_grib_get(message, "Ny"),
                    "dx_metres": safe_grib_get(message, "DxInMetres"),
                    "dy_metres": safe_grib_get(message, "DyInMetres"),
                    "bitmap_present": safe_grib_get(message, "bitmapPresent"),
                    "missing_value": safe_grib_get(message, "missingValue"),
                    }

                selected_data[field_name] = {"metadata": metadata,
                                             "values": codes_get_array(message, "values")}

            codes_release(message)

    missing_fields = set(selected_field_rules) - set(selected_data)

    if missing_fields:
        raise ValueError(f"Selected fields were not read: {sorted(missing_fields)}")

    return selected_data

def get_valid_values(field_data):
    """
    Separate valid and missing values using the GRIB metadata.

    """
    
    values = field_data["values"].astype("float64")
    metadata = field_data["metadata"]

    grib_missing_mask = np.zeros(values.shape, dtype=bool)
    
    if metadata["bitmap_present"] == 1:
        grib_missing_mask = (values == metadata["missing_value"])

    invalid_number_mask = ~np.isfinite(values)

    # lets combine this with infinity missing mask
    excluded_mask = (grib_missing_mask | invalid_number_mask)

    return {
        "all_values": values,
        "valid_values": values[~excluded_mask],
        "grib_missing_mask": grib_missing_mask,
        "invalid_number_mask": invalid_number_mask,
        "excluded_mask": excluded_mask,
        }


def summarize_values(name, values, excluded_mask, units):
    """
    Return a compact summary of valid field values.

    """
    
    valid_values = values[~excluded_mask]

    return {
        "field": name,
        "units": units,
        "grib_missing_cells": int(
            selected_masks[name]["grib_missing"].sum()
        ),
        "invalid_number_cells": int(
            selected_masks[name]["invalid_numbers"].sum()
        ),
        "minimum": np.min(valid_values),
        "median": np.median(valid_values),
        "95th_percentile": np.percentile(valid_values, 95),
        "99th_percentile": np.percentile(valid_values, 99),
        "maximum": np.max(valid_values),
        }
    



### **Introduction**

The High-Resolution Rapid Refresh (HRRR) is NOAA's short-range weather model for the continental United States. It runs every hour on an approximately 3-km grid and uses recent radar, satellite, surface, aircraft, and other observations to estimate the current atmosphere and produce forecasts. Radar observations are assimilated during model initialization, making HRRR useful for representing rapidly changing storms [1, 2].

HRRR provides several output products:

- **Surface (`wrfsfc`)**: two-dimensional fields such as precipitation, composite reflectivity, temperature, humidity, wind, precipitable water, and CAPE.
- **Pressure-level (`wrfprs`)**: atmospheric variables at standard pressure levels.
- **Native-level (`wrfnat`)**: detailed variables on the model's original vertical levels.
- **Subhourly (`wrfsubh`)**: selected fields at 15-minute intervals.
- **Smoke and Alaska products**: specialized products outside the scope of this study.


We will be using **HRRR CONUS hourly surface product (`wrfsfc`)**. It contains the precipitation and atmospheric fields relevant to hourly QPE correction without the size and complexity of the full three-dimensional products.

We will be using HRRR as predictor information rather than observed rainfall truth. It can describe the model predicted precipitation and the surrounding atmospheric environment. These fields may help the correction model recognize conditions where radar-only QPE has systematic errors.

The main datasets will have different roles:
- MRMS RadarOnly QPE: radar-estimated hourly precipitation.
- RAQI: radar-precipitation quality information.
- HRRR: predicted precipitation and atmospheric conditions.
- MADIS/CoCoRaHS: gauge observations used to train and evaluate the correction.
  
There is one more thing: forecast product is also suitable for real time monitoring because it predicts conditions for upcoming hours. In an operational system, we can use newest HRRR cycle that was available before the correction was produced.

In the next sections, we will download a small sample, inspect its variables and metadata, verify the precipitation accumulation window, study the distributions, align the HRRR grid and time with MRMS and gauges, and visualize selected fields before downloading the complete archive.

### **Data Download**

We will download archived HRRR data from NOAA's public AWS archive. The archive does not require an AWS account, and the files can be accessed directly through HTTPS.

For each HRRR model cycle, we will use the CONUS surface product at forecast hour 1:

```text
hrrr.YYYYMMDD/conus/hrrr.tCCz.wrfsfcf01.grib2
```

Here, CC is the initialization hour in UTC. For example:

```text
    hrrr.20240701/conus/hrrr.t18z.wrfsfcf01.grib2
```

is initialized at 18:00 UTC on July 1, 2024, and is valid at 19:00 UTC. Its one hour accumulated precipitation represents the model forecast for 18:00-19:00 UTC.

The files use the GRIB2 format. A GRIB2 file contains many weather fields on the same HRRR grid, including accumulated precipitation, composite reflectivity, temperature, humidity, wind, precipitable water, and CAPE. Each file also has a small .idx file listing the GRIB messages and their byte locations.

A complete initialization day has 24 forecast-hour-1 files, one from each hourly HRRR cycle. However, these files do not exactly cover one UTC valid-time day, for example:

```text
00 UTC cycle + f01 → valid at 01 UTC
...
23 UTC cycle + f01 → valid at 00 UTC on the following day
```


Now lets work on downloading a specific day data and inspect it. 


In [24]:

# we will select a random date between our start and end dates to download the data
start_date = date(2021, 1, 1)
end_date = date(2025, 12, 31)

random_generator = random.Random(14912) # in case if we want to fix the date for the reproducibility

number_of_days = (end_date - start_date).days
random_offset = random_generator.randint(0, number_of_days)

# now lets select the date and change the format to Ymd
selected_date = start_date + timedelta(days=random_offset)

download_date = selected_date.strftime("%Y%m%d")
forecast_hour = 1

# now lets get the path - we will create the directory by the date so each day should have its own directory
hrrr_root = data_path / "raw_data" 
day_directory = hrrr_root / download_date
day_directory.mkdir(parents=True, exist_ok=True)


# lets also define the aws base url to get the data from the AWS
aws_base_url = "https://noaa-hrrr-bdp-pds.s3.amazonaws.com"

# we will select a first cycle just for inspecting the data: later we will use range(24) once we verified .
cycles = [0]

print("selected date:", selected_date)
print("download directory:", day_directory)
      

selected date: 2023-06-28
download directory: /home/adil/Desktop/PhD_Research/Research_Projects/QPE_Project/dataset/data_understanding/hrrr/data/raw_data/20230628


In [25]:

# lets construct the source urls now for the relevant day data
def hrrr_filename(cycle_hour, forecast_hour=1):
    """Return the HRRR surface filename for one cycle and lead time."""
    
    return f"hrrr.t{cycle_hour:02d}z.wrfsfcf{forecast_hour:02d}.grib2"


def hrrr_url(date, cycle_hour, forecast_hour=1):
    """Return the public AWS URL for one HRRR surface file."""
    
    filename = hrrr_filename(cycle_hour, forecast_hour)
    return f"{aws_base_url}/hrrr.{date}/conus/{filename}"

# now we can loop through cycles and get the url for each cycle
for cycle in cycles:
    print(hrrr_url(download_date, cycle, forecast_hour))

# if we click on the link below it should start downloading the file


https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20230628/conus/hrrr.t00z.wrfsfcf01.grib2


In [26]:

# now lets download the files
downloaded_files = []

for cycle in cycles:
    filename = hrrr_filename(cycle, forecast_hour)
    grib_url = hrrr_url(download_date, cycle, forecast_hour)
    grib_path = day_directory / filename

    downloaded_files.append(download_file(grib_url, grib_path))
    download_file(grib_url + ".idx", day_directory / f"{filename}.idx")

print(f"\nDownloaded {len(downloaded_files)} GRIB2 file(s).")


Already downloaded: hrrr.t00z.wrfsfcf01.grib2 (154.9 MB)
Already downloaded: hrrr.t00z.wrfsfcf01.grib2.idx (0.0 MB)

Downloaded 1 GRIB2 file(s).


### **Data Inspection**

Before reading the HRRR data, we will confirm the identity and basic integrity of the downloaded files. We will check the filenames, sizes, initialization time, forecast lead, valid time, source URL, and the presence of the GRIB2 and index files.

A valid GRIB2 file normally begins with the bytes `GRIB` and ends with `7777`.


In [38]:

# get the file records, create pd and inspect the pd
file_records = [inspect_downloaded_file(grib_path) for grib_path in downloaded_files]

file_summary = pd.DataFrame(file_records)
display(file_summary)

# in case if we want to print it
for record in file_records:
    print("filename:", record["filename"])
    print("initialization time:", record["initialization_time"])
    print("forecast hour:", record["forecast_hour"])
    print("valid time:", record["valid_time"])
    print("GRIB2 size:", f'{record["grib_size_mb"]:.1f} MB')
    print("index size:", f'{record["index_size_kb"]:.1f} KB')
    print("index messages:", record["index_message_count"])
    print("starts with GRIB:", record["starts_with_grib"])
    print("ends with 7777:", record["ends_with_7777"])
    print("basic integrity passed:", record["basic_integrity_passed"])
    print("source:", record["source_url"])
    print("local path:", record["local_path"])


,filename,cycle_hour_utc,forecast_hour,initialization_time,valid_time,grib_exists,index_exists,grib_size_mb,index_size_kb,index_message_count,starts_with_grib,ends_with_7777,basic_integrity_passed,source_url,local_path
0,hrrr.t00z.wrfsfcf01.grib2,0,1,2023-06-28 00:00:00+00:00,2023-06-28 01:00:00+00:00,True,True,154.935962,10.18,170,True,True,True,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...,/home/adil/Desktop/PhD_Research/Research_Proje...


filename: hrrr.t00z.wrfsfcf01.grib2
initialization time: 2023-06-28 00:00:00+00:00
forecast hour: 1
valid time: 2023-06-28 01:00:00+00:00
GRIB2 size: 154.9 MB
index size: 10.2 KB
index messages: 170
starts with GRIB: True
ends with 7777: True
basic integrity passed: True
source: https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrrr.20230628/conus/hrrr.t00z.wrfsfcf01.grib2
local path: /home/adil/Desktop/PhD_Research/Research_Projects/QPE_Project/dataset/data_understanding/hrrr/data/raw_data/20230628/hrrr.t00z.wrfsfcf01.grib2



A GRIB2 file contains many weather fields. Its `.idx` file provides a compact inventory of those fields without requiring us to decode the complete GRIB2 file.

We inspect the index for two reasons:

1. To identify the exact variable, level, and time definition of each field.
2. To obtain the byte range of each field so that future downloads can retrieve only the variables we need.
   

In [53]:

# get the path
sample_grib_path = Path(downloaded_files[0])
sample_index_path = Path(str(sample_grib_path) + ".idx")

index_table = parse_hrrr_index(sample_index_path, sample_grib_path)
print("number of GRIB messages:", len(index_table))

index_table.head()
    

number of GRIB messages: 170


,message,start_byte,reference_time,variable,level,time_description,original_index_line,end_byte,message_size_mb
0,1,0,2023062800,REFC,entire atmosphere,1 hour fcst,1:0:d=2023062800:REFC:entire atmosphere:1 hour...,429810,0.429811
1,2,429811,2023062800,RETOP,cloud top,1 hour fcst,2:429811:d=2023062800:RETOP:cloud top:1 hour f...,647574,0.217764
2,3,647575,2023062800,var discipline=0 center=7 local_table=1 parmca...,entire atmosphere,1 hour fcst,3:647575:d=2023062800:var discipline=0 center=...,1025484,0.377910
3,4,1025485,2023062800,VIL,entire atmosphere,1 hour fcst,4:1025485:d=2023062800:VIL:entire atmosphere:1...,1255269,0.229785
4,5,1255270,2023062800,VIS,surface,1 hour fcst,5:1255270:d=2023062800:VIS:surface:1 hour fcst:,2573893,1.318624


In [55]:

# lets create compact inventory of available fields - a variable code alone is not enough 
# to identify a field. For example, temperature may occur at 2 m above ground, 
# at the surface, or at another atmospheric level. we therefore need the unique combination of variable, level 
# and time description - Tthis tells us exactly which versions of each field are available.
field_inventory = (index_table[[ "variable", "level", "time_description"]]
                    .drop_duplicates()
                    .sort_values(["variable", "level", "time_description"]).reset_index(drop=True))

print("unique field definitions:", len(field_inventory))
field_inventory


unique field definitions: 170


,variable,level,time_description
0,4LFTX,180-0 mb above ground,1 hour fcst
1,AOTK,entire atmosphere (considered as a single layer),1 hour fcst
2,APCP,surface,0-1 hour acc fcst
3,ASNOW,surface,0-1 hour acc fcst
4,BGRUN,surface,0-1 hour acc fcst
...,...,...,...
165,VVCSH,0-6000 m above ground,1 hour fcst
166,WEASD,surface,0-1 hour acc fcst
167,WEASD,surface,1 hour fcst
168,WIND,10 m above ground,0-1 hour max fcst


In [56]:

# no we will inspect only the candidate fields -we do not need every HRRR field. 
# at this stage, we only need to confirm the availability and exact descriptions of variables 
# that may help with precipitation correction.
candidate_variables = [
    "APCP",
    "REFC",
    "PWAT",
    "TMP",
    "DPT",
    "RH",
    "UGRD",
    "VGRD",
    "CAPE",
    "CIN",
    ]

candidate_fields = field_inventory[field_inventory["variable"]
                        .isin(candidate_variables)].reset_index(drop=True)

candidate_fields


,variable,level,time_description
0,APCP,surface,0-1 hour acc fcst
1,CAPE,0-3000 m above ground,1 hour fcst
2,CAPE,180-0 mb above ground,1 hour fcst
3,CAPE,255-0 mb above ground,1 hour fcst
4,CAPE,90-0 mb above ground,1 hour fcst
5,CAPE,surface,1 hour fcst
6,CIN,180-0 mb above ground,1 hour fcst
7,CIN,255-0 mb above ground,1 hour fcst
8,CIN,90-0 mb above ground,1 hour fcst
9,CIN,surface,1 hour fcst


In [57]:

# next - inspect accumulated precipitation separately
# APCP is our first priority because its accumulation interval must match the hourly MRMS rainfall window.
# for an f01 file, we expect surface precipitation accumulated from forecast hour 0 through forecast hour 1.
apcp_records = index_table[index_table["variable"].eq("APCP")][
                        [
                        "message",
                        "variable",
                        "level",
                        "time_description",
                        "start_byte",
                        "end_byte",
                        "message_size_mb",
                        "original_index_line",
                        ]].reset_index(drop=True)

apcp_records


,message,variable,level,time_description,start_byte,end_byte,message_size_mb,original_index_line
0,84,APCP,surface,0-1 hour acc fcst,60532967,60756207,0.223241,84:60532967:d=2023062800:APCP:surface:0-1 hour...


In [58]:

# lastly - estimate the size of the candidate fields - complete surface file is large. 
# before designing selective downloads, we need to know how much data the 
# candidate records occupy compared with the complete file. this is only an initial estimate 
# because we have not finalized the variable list.
candidate_records = index_table[index_table["variable"].isin(candidate_variables)].copy()

complete_file_size_mb = sample_grib_path.stat().st_size / 1e6
candidate_size_mb = candidate_records["message_size_mb"].sum()

print(f"complete surface file: {complete_file_size_mb:.1f} MB")
print(f"all candidate records: {candidate_size_mb:.1f} MB")
print(
        "candidate fraction:",
        f"{100 * candidate_size_mb / complete_file_size_mb:.1f}%")



complete surface file: 154.9 MB
all candidate records: 41.5 MB
candidate fraction: 26.8%



The downloaded file is the HRRR CONUS surface product initialized at 00:00 UTC on June 28, 2023, with a one-hour forecast valid at 01:00 UTC (might change if we select the random another file). Index confirms that `APCP` is surface total precipitation accumulated from forecast hour 0 through hour 1. Its time window therefore matches an hourly MRMS accumulation ending at the HRRR valid time.

The surface product also contains near-surface atmospheric fields that may help explain precipitation errors. Our initial candidate fields are 2-m temperature, 2-m relative humidity, and 10-m wind components. These fields are not yet confirmed model inputs, their value will be evaluated after inspecting their metadata, distributions, and relationship with precipitation.




Lets proceed to the next steps, this is where we will inspect the individual fields. First lets select only the relevant fields . Earlier we saw that the same variable can appear at several vertical levels, for example, temperature is available at 2 m above ground and at several pressure levels. Therefore we must select the exact combination of variable, level, and time description that represents the field we intend to use.


In [86]:

selected_field_rules = {
    "apcp": {
        "variable": "APCP",
        "level": "surface",
        "time_description": "0-1 hour acc fcst",
        },
    "temperature_2m": {
        "variable": "TMP",
        "level": "2 m above ground",
        "time_description": "1 hour fcst",
        },
    "relative_humidity_2m": {
        "variable": "RH",
        "level": "2 m above ground",
        "time_description": "1 hour fcst",
        },
    }

# lets print them
selected_records = []

for field_name, rule in selected_field_rules.items():
    matches = index_table[
        index_table["variable"].eq(rule["variable"])
        & index_table["level"].eq(rule["level"])
        & index_table["time_description"].eq(rule["time_description"])
        ].copy()

    if len(matches) != 1:
        raise ValueError(f"{field_name}: expected one index record, found {len(matches)}")

    matches["field_name"] = field_name
    selected_records.append(matches)

selected_records = pd.concat(selected_records, ignore_index=True)

selected_records[
                [
                "field_name",
                "message",
                "variable",
                "level",
                "time_description",
                "message_size_mb",
                ]
                ]


,field_name,message,variable,level,time_description,message_size_mb
0,apcp,84,APCP,surface,0-1 hour acc fcst,0.223241
1,temperature_2m,71,TMP,2 m above ground,1 hour fcst,1.254497
2,relative_humidity_2m,75,RH,2 m above ground,1 hour fcst,1.510615


In [87]:

# now that we know at what message we have what variable, lets use that informatino t oread the selected grib 
# messages - we will read only the five selected messages, not all 170 fields.
selected_data = read_selected_messages(sample_grib_path,  selected_records)
selected_data


{'temperature_2m': {'metadata': {'field_name': 'temperature_2m',
   'name': '2 metre temperature',
   'short_name': '2t',
   'units': 'K',
   'type_of_level': 'heightAboveGround',
   'level': 2,
   'step_type': 'instant',
   'start_step': 1,
   'end_step': 1,
   'initialization_date': 20230628,
   'initialization_time': 0,
   'validity_date': 20230628,
   'validity_time': 100,
   'grid_type': 'lambert',
   'nx': 1799,
   'ny': 1059,
   'dx_metres': 3000.0,
   'dy_metres': 3000.0,
   'bitmap_present': 0,
   'missing_value': 9999},
  'values': array([293.66516113, 293.66516113, 293.66516113, ..., 283.35266113,
         283.29016113, 283.29016113], shape=(1905141,))},
 'relative_humidity_2m': {'metadata': {'field_name': 'relative_humidity_2m',
   'name': '2 metre relative humidity',
   'short_name': '2r',
   'units': '%',
   'type_of_level': 'heightAboveGround',
   'level': 2,
   'step_type': 'instant',
   'start_step': 1,
   'end_step': 1,
   'initialization_date': 20230628,
   'initiali

In [88]:

# now we can inspect the important metadata, such as what does each field represents, what are the units,
# is it accumulated? is it valid and if yes when it is valid?
metadata_table = pd.DataFrame([field_data["metadata"] for field_data in selected_data.values()])
metadata_table


,field_name,name,short_name,units,type_of_level,level,step_type,start_step,end_step,initialization_date,initialization_time,validity_date,validity_time,grid_type,nx,ny,dx_metres,dy_metres,bitmap_present,missing_value
0,temperature_2m,2 metre temperature,2t,K,heightAboveGround,2,instant,1,1,20230628,0,20230628,100,lambert,1799,1059,3000.0,3000.0,0,9999
1,relative_humidity_2m,2 metre relative humidity,2r,%,heightAboveGround,2,instant,1,1,20230628,0,20230628,100,lambert,1799,1059,3000.0,3000.0,0,9999
2,apcp,Total Precipitation,tp,kg m**-2,surface,0,accum,0,1,20230628,0,20230628,100,lambert,1799,1059,3000.0,3000.0,0,9999


In [89]:

# now we can inspect the values. lets start with the precipitation values
apcp_data = get_valid_values(selected_data["apcp"])
apcp_values = apcp_data["all_values"]
valid_apcp = apcp_data["valid_values"]

print("GRIB missing cells:", apcp_data["grib_missing_mask"].sum())
print( "NaN or infinite cells:", apcp_data["invalid_number_mask"].sum())

# get the units
apcp_units = selected_data["apcp"]["metadata"]["units"]

if apcp_units != "kg m**-2":
    raise ValueError(f"Unexpected APCP units: {apcp_units}")


GRIB missing cells: 0
NaN or infinite cells: 0


In [90]:



# now we know that one kg of water per square metre equals one millimetre of water depth.
apcp_mm = valid_apcp
positive_apcp_mm = apcp_mm[apcp_mm > 0]

apcp_summary = pd.Series(
            {
            "units_after_interpretation": "mm",
            "total_cells": apcp_values.size,
            "missing_cells": apcp_data["grib_missing_mask"].sum(),
            "negative_cells": int(np.count_nonzero(apcp_mm < 0)),
            "zero_cells": int(np.count_nonzero(apcp_mm == 0)),
            "positive_cells": int(positive_apcp_mm.size),
            "positive_cell_fraction_percent": (100 * positive_apcp_mm.size / apcp_mm.size),
            "minimum_mm": np.min(apcp_mm),
            "median_positive_mm": (
                np.median(positive_apcp_mm)
                if positive_apcp_mm.size
                else np.nan
                ),
            "95th_positive_percentile_mm": (
                np.percentile(positive_apcp_mm, 95)
                if positive_apcp_mm.size
                else np.nan
                ),
            "99th_positive_percentile_mm": (
                np.percentile(positive_apcp_mm, 99)
                if positive_apcp_mm.size
                else np.nan
                ),
            "maximum_mm": np.max(apcp_mm),
            },
            name="value",
            )

apcp_summary


units_after_interpretation              mm
total_cells                        1905141
missing_cells                            0
negative_cells                           0
zero_cells                         1740651
positive_cells                      164490
positive_cell_fraction_percent    8.634007
minimum_mm                             0.0
median_positive_mm                    0.08
95th_positive_percentile_mm           4.17
99th_positive_percentile_mm          20.63
maximum_mm                          118.38
Name: value, dtype: object

#### **Inspecting Atmospheric Fields**


Lets inspect the variables such as temperature, relative humidity and wind


In [91]:

temperature_data = get_valid_values(selected_data["temperature_2m"])
humidity_data = get_valid_values(selected_data["relative_humidity_2m"])

temperature_values = temperature_data["all_values"]
humidity_values = humidity_data["all_values"]

temperature_excluded = temperature_data["excluded_mask"]
humidity_excluded = humidity_data["excluded_mask"]

expected_units = {
    "temperature_2m": "K",
    "relative_humidity_2m": "%",
    }

for field_name, expected_unit in expected_units.items():
    actual_unit = selected_data[field_name]["metadata"]["units"]

    if actual_unit != expected_unit:
        raise ValueError(
            f"{field_name}: expected {expected_unit}, "
            f"found {actual_unit}"
        )

print("All selected atmospheric fields have the expected units.")


All selected atmospheric fields have the expected units.


In [92]:

temperature_c = temperature_values - 273.15
relative_humidity_percent = humidity_values


selected_masks = {
    "2-m temperature": {
        "grib_missing": temperature_data["grib_missing_mask"],
        "invalid_numbers": temperature_data["invalid_number_mask"],
        },
    "2-m relative humidity": {
        "grib_missing": humidity_data["grib_missing_mask"],
        "invalid_numbers": humidity_data["invalid_number_mask"],
        },
        }

# now lets create the summary
candidate_summary = pd.DataFrame(
    [
        summarize_values(
            "2-m temperature",
            temperature_c,
            temperature_excluded,
            "°C",
        ),
        summarize_values(
            "2-m relative humidity",
            relative_humidity_percent,
            humidity_excluded,
            "%",
        ),
    ]
)

candidate_summary


,field,units,grib_missing_cells,invalid_number_cells,minimum,median,95th_percentile,99th_percentile,maximum
0,2-m temperature,°C,0,0,-0.672339,24.015161,35.327661,38.702661,41.702661
1,2-m relative humidity,%,0,0,2.300000,72.200000,96.300000,99.000000,100.000000



The HRRR file contains 1,905,141 cells, matching its 1799 × 1059 grid. APCP, 2-m temperature, and 2-m relative humidity contain no GRIB missing values, non-finite values, or physically invalid values in this sample.

APCP is expressed in `kg m⁻²`, which is numerically equivalent to millimetres of water. Approximately 8.63% of the CONUS cells contain positive precipitation. Among wet cells, the median is 0.08 mm, the 95th percentile is 4.17 mm, and the 99th percentile is 20.63 mm. The distribution is strongly right-skewed, with a maximum of 118.38 mm.

The 2-m temperature ranges from −0.67°C to 41.70°C, while 2-m relative humidity ranges from 2.3% to 100%. Both fields have physically plausible values for the complete CONUS domain.


### **Reference and Documentation**

1. [NOAA Global Systems Laboratory: High-Resolution Rapid Refresh](https://rapidrefresh.noaa.gov/hrrr/)
2. [NOAA Environmental Modeling Center: HRRR System Description](https://emc.ncep.noaa.gov/emc/pages/numerical_forecast_systems/hrrr.php)
3. [NCEP: HRRR Product Inventory](https://www.nco.ncep.noaa.gov/pmb/products/hrrr/)
4. [NOAA HRRR Archive on AWS](https://registry.opendata.aws/noaa-hrrr-pds/)